# 6. Reference-Based Segmentation Comparison

**Purpose:** Compare segmentation methods against a scRNA-seq reference atlas using metrics
from [Segger](https://github.com/EliHei2/segger_dev).

**What this measures:**
- **MECR** — Mutually Exclusive Co-expression Rate (lower = cleaner segmentation)
- **Contamination** — How much marker signal leaks into neighboring cells of different types
- **Sensitivity** — How well each method detects known cell-type markers
- **Clustering scores** — Calinski-Harabasz and Silhouette on annotated cell types
- **Neighborhood entropy** — Spatial mixing of cell types

**Requires:**
- Completed segmentation runs (h5ad files with coarse cell type annotations)
- A scRNA-seq reference atlas (h5ad) for marker discovery

**Container:** Must be run inside `seg_sin_V1.sif`.

In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sc.settings.set_figure_params(dpi=100, frameon=False)
%matplotlib inline

# Add scripts to path so we can import segger_functions
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

from segger_functions import (
    find_markers,
    find_mutually_exclusive_genes,
    compute_MECR,
    compute_quantized_mecr_area,
    compute_quantized_mecr_counts,
    calculate_contamination,
    calculate_sensitivity,
    compute_clustering_scores,
    compute_neighborhood_metrics,
    compute_transcript_density,
    plot_mecr_results,
    plot_quantized_mecr_area,
    plot_quantized_mecr_counts,
    plot_contamination_boxplots,
    plot_contamination_heatmaps,
    plot_entropy_boxplots,
    plot_sensitivity_boxplots,
    plot_cell_counts,
    plot_counts_per_cell,
    plot_cell_area,
    plot_transcript_density,
    plot_general_statistics_plots,
)

print('Segger functions loaded')

## Configuration

Edit these paths to match your setup.

In [ ]:
# ── Paths ──
# Reference scRNA-seq atlas
REFERENCE_H5AD = "/path/to/reference_atlas.h5ad"
REFERENCE_CELLTYPE_COL = "paperLabels"  # column in reference .obs with cell type labels

# Segmentation results — load the annotated h5ad files
# These should already have 'coarse_annot' in .obs from your annotation step
SEGMENTATION_H5ADS = {
    "10x":    "/path/to/10x_annotated.h5ad",
    "proseg": "/path/to/proseg_annotated.h5ad",
    "baysor": "/path/to/baysor_annotated.h5ad",
    # Add more methods as needed:
    # "cellpose": "/path/to/cellpose_annotated.h5ad",
}

# Column name for coarse cell type annotations in the segmentation h5ads
CELLTYPE_COL = "coarse_annot"

# Output directory for figures
OUTPUT_PATH = Path("segger_comparison_figs")
OUTPUT_PATH.mkdir(exist_ok=True)

# Color palette for methods
METHOD_COLORS = {
    "10x":      "#E6D6FF",
    "baysor":   "#A06CD5",
    "proseg":   "#5B2A86",
    "cellpose": "#2E1A47",
}

## Load reference atlas

In [ ]:
reference = sc.read_h5ad(REFERENCE_H5AD)
print(f"Reference: {reference.n_obs:,} cells × {reference.n_vars:,} genes")

# ── Map reference cell types to coarse annotations ──
# Edit this mapping to match YOUR reference atlas labels → your coarse categories
cell_type_mapping = {
    # T cells
    'abT (CD4)': 'Tcell_CD4', 'Treg': 'Tcell_CD4', 'Th17': 'Tcell_CD4', 'MAIT': 'Tcell_CD4',
    'abT (CD8)': 'Tcell_CD8', 'NK': 'Tcell_CD8', 'gd T': 'Tcell_CD8',
    # B cells
    'B': 'Bcell', 'Plasma': 'Plasmacells',
    # Myeloid
    'Monocyte': 'Mac/mono', 'mDC': 'mDC', 'pDC': 'pDC', 'Neutrophil': 'Mac/mono', 'Mast': 'MAST',
    # Fibroblasts
    'H.Fib 1.1': 'Fibro', 'H.Fib 1.2': 'Fibro', 'H.Fib 1.3': 'Fibro',
    'H.Fib 1.4': 'Fibro', 'H.Fib 1.5': 'Fibro',
    # Epithelium
    'H.Epi 1.1': 'Epi', 'H.Epi 1.2': 'Epi', 'H.Epi 1.3': 'Epi',
    # Endothelium
    'H.VEC 1.1': 'Endo_vascular', 'H.VEC 1.2': 'Endo_vascular',
    'H.VEC 1.3': 'Endo_vascular', 'H.VEC 1.4': 'Endo_vascular',
    'H.LEC': 'Endo_lymph', 'H.SMC': 'Endo_smooth_muscle',
    # Other
    'H.Mel': 'melano', 'Other': 'Endo',
}

reference.obs['coarse_annotation'] = (
    reference.obs[REFERENCE_CELLTYPE_COL].map(cell_type_mapping).fillna('Other')
)
print(f"Coarse types: {reference.obs['coarse_annotation'].nunique()}")
reference.obs['coarse_annotation'].value_counts()

## Load segmentation results

In [ ]:
segment_dict = {}
for method, path in SEGMENTATION_H5ADS.items():
    adata = sc.read_h5ad(path)
    print(f"{method:12s}: {adata.n_obs:>8,} cells × {adata.n_vars:>5,} genes")
    segment_dict[method] = adata

# ── Standardize column names across methods ──
for method, adata in segment_dict.items():
    # Ensure cell_area exists
    if 'cell_area' not in adata.obs.columns and 'area' in adata.obs.columns:
        adata.obs['cell_area'] = adata.obs['area']
    # Ensure transcripts column exists
    if 'transcripts' not in adata.obs.columns and 'n_counts' in adata.obs.columns:
        adata.obs['transcripts'] = adata.obs['n_counts']
    # Ensure celltype_major exists (needed by some segger functions)
    if 'celltype_major' not in adata.obs.columns and CELLTYPE_COL in adata.obs.columns:
        adata.obs['celltype_major'] = adata.obs[CELLTYPE_COL].copy()
    # Ensure raw counts layer
    if 'raw' not in adata.layers and 'counts' in adata.layers:
        adata.layers['raw'] = adata.layers['counts'].copy()
    # Ensure spatial centroids in .obs
    if 'cell_centroid_x' not in adata.obs.columns and 'spatial' in adata.obsm:
        adata.obs['cell_centroid_x'] = adata.obsm['spatial'][:, 0]
        adata.obs['cell_centroid_y'] = adata.obsm['spatial'][:, 1]

print(f"\nLoaded {len(segment_dict)} method(s)")

## Discover markers from reference

In [ ]:
# Intersect genes between reference and segmentation panels
seg_genes = list(set().union(*[set(a.var_names) for a in segment_dict.values()]))
reference_filtered = reference[:, reference.var_names.isin(seg_genes)].copy()
print(f"Reference filtered to {reference_filtered.n_vars} overlapping genes")

# Find cell-type markers
markers = find_markers(
    reference_filtered,
    cell_type_column="coarse_annotation",
    pos_percentile=30,
    neg_percentile=5,
)

# Find mutually exclusive gene pairs
exclusive_gene_pairs = find_mutually_exclusive_genes(
    adata=reference_filtered,
    markers=markers,
    cell_type_column='coarse_annotation',
)
print(f"Found {len(exclusive_gene_pairs)} mutually exclusive gene pairs")

---
## 1. General Statistics

In [ ]:
plot_general_statistics_plots(segment_dict, OUTPUT_PATH, palette=METHOD_COLORS)

## 2. MECR — Mutually Exclusive Co-expression Rate

Lower MECR = less co-expression of genes that should be exclusive to different cell types = cleaner segmentation.

In [ ]:
# Compute MECR for each method
mecr_results = {}
for method, adata in segment_dict.items():
    mecr_results[method] = compute_MECR(adata, exclusive_gene_pairs)

plot_mecr_results(mecr_results, output_path=OUTPUT_PATH, palette=METHOD_COLORS)

In [ ]:
# MECR vs cell area and transcript counts
quantized_mecr_area = {}
quantized_mecr_counts = {}

for method, adata in segment_dict.items():
    if 'cell_area' in adata.obs.columns:
        quantized_mecr_area[method] = compute_quantized_mecr_area(
            adata=adata, gene_pairs=exclusive_gene_pairs
        )
    quantized_mecr_counts[method] = compute_quantized_mecr_counts(
        adata=adata, gene_pairs=exclusive_gene_pairs
    )

if quantized_mecr_area:
    plot_quantized_mecr_area(quantized_mecr_area, output_path=OUTPUT_PATH, palette=METHOD_COLORS)
plot_quantized_mecr_counts(quantized_mecr_counts, output_path=OUTPUT_PATH, palette=METHOD_COLORS)

## 3. Contamination Analysis

How much cell-type-specific marker expression leaks into neighboring cells of different types.

In [ ]:
contamination_results = {}
for method, adata in segment_dict.items():
    if 'cell_centroid_x' in adata.obs.columns:
        print(f"Computing contamination for {method}...")
        contamination_results[method] = calculate_contamination(
            adata=adata,
            markers=markers,
            radius=15,
            n_neighs=20,
            celltype_column='celltype_major',
            num_cells=1000,
        )

if contamination_results:
    # Heatmaps
    plot_contamination_heatmaps(contamination_results, cmap='Purples', vmax=1)

    # Boxplots
    boxplot_data = []
    for method, df in contamination_results.items():
        melted = df.reset_index().melt(
            id_vars=['Source Cell Type'],
            var_name='Target Cell Type',
            value_name='Contamination',
        )
        melted['Segmentation Method'] = method
        boxplot_data.append(melted)
    boxplot_data = pd.concat(boxplot_data)
    plot_contamination_boxplots(boxplot_data, OUTPUT_PATH, palette=METHOD_COLORS)

## 4. Clustering Scores

In [ ]:
clustering_scores = {}
for method, adata in segment_dict.items():
    ch, sh = compute_clustering_scores(adata, cell_type_column='celltype_major')
    clustering_scores[method] = (ch, sh)
    print(f"{method:12s}: CH={ch:.1f}, Silhouette={sh:.4f}")

# Bar plot
clust_df = pd.DataFrame.from_dict(
    clustering_scores, orient='index', columns=['CH Score', 'Silhouette Score']
).reset_index().rename(columns={'index': 'Method'})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ['CH Score', 'Silhouette Score']):
    colors = [METHOD_COLORS.get(m, 'gray') for m in clust_df['Method']]
    ax.bar(clust_df['Method'], clust_df[metric], color=colors)
    ax.set_title(metric)
    ax.set_ylabel('Score')
    for i, v in enumerate(clust_df[metric]):
        ax.text(i, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'clustering_scores.pdf', bbox_inches='tight')
plt.show()

## 5. Neighborhood Entropy

Higher entropy = more mixing of cell types in the local neighborhood.

In [ ]:
for method, adata in segment_dict.items():
    if 'spatial' in adata.obsm:
        print(f"Computing neighborhood metrics for {method}...")
        compute_neighborhood_metrics(
            adata, radius=15, celltype_column='celltype_major', n_neighs=20
        )

entropy_data = []
for method, adata in segment_dict.items():
    if 'neighborhood_entropy' in adata.obs.columns:
        df = pd.DataFrame({
            'Cell Type': adata.obs['celltype_major'],
            'Neighborhood Entropy': adata.obs['neighborhood_entropy'],
            'Segmentation Method': method,
        }).dropna(subset=['Neighborhood Entropy'])
        entropy_data.append(df)

if entropy_data:
    entropy_data = pd.concat(entropy_data)
    plot_entropy_boxplots(entropy_data, OUTPUT_PATH, palette=METHOD_COLORS)

## 6. Marker Sensitivity

What fraction of known markers for each cell type are actually detected in the segmentation?

In [ ]:
purified_markers = find_markers(
    reference_filtered, 'coarse_annotation', pos_percentile=20, percentage=75
)

sensitivity_data = []
for method, adata in segment_dict.items():
    results = calculate_sensitivity(adata, purified_markers, max_cells_per_type=2000)
    for cell_type, sensitivities in results.items():
        df = pd.DataFrame({
            'Cell Type': cell_type,
            'Sensitivity': sensitivities,
            'Segmentation Method': method,
        })
        sensitivity_data.append(df)

sensitivity_data = pd.concat(sensitivity_data)
plot_sensitivity_boxplots(sensitivity_data, OUTPUT_PATH, palette=METHOD_COLORS)

---

## Summary

All figures are saved to the output directory. Key files:
- `mecr_results_boxplot.pdf` — MECR comparison
- `contamination_boxplots.pdf` — Contamination by cell type
- `clustering_scores.pdf` — CH + Silhouette
- `neighborhood_entropy_boxplots.pdf` — Spatial mixing
- `sensitivity_boxplots.pdf` — Marker detection
- `general_statistics_plots.pdf` — Cell counts, areas, densities

CSV data for each plot is also saved alongside the PDFs for downstream use.